In [0]:
import pandas as pd
from pyspark.sql import functions as F

df_silver = spark.table("internet_fijo_elt.silver.conexiones_internet_fijo")

display(df_silver.limit(10))


In [0]:
print("Registros Silver:", df_silver.count())

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS internet_fijo_elt.gold")

In [0]:
dim_periodo = (
    df_silver
    .select("periodo")
    .distinct()
    .withColumn("fecha_id", F.date_format("periodo", "yyyyMM").cast("int"))
    .withColumn("año", F.year("periodo"))
    .withColumn("mes", F.month("periodo"))
    .withColumn("trimestre", F.quarter("periodo"))
    .withColumn("nombre_mes", F.date_format("periodo", "MMMM"))
    .select("fecha_id", "periodo", "año", "mes", "nombre_mes", "trimestre")
)


In [0]:
dim_periodo.write.format("delta").mode("overwrite").saveAsTable(
    "internet_fijo_elt.gold.dim_periodo"
)

In [0]:
print("Dim Periodo:", dim_periodo.count())

In [0]:
display(dim_periodo.orderBy("periodo"))